# HW3 — Two retrievers and a model: what RAG actually sends

**Name: Sairanbek Darkhan**
**Student ID: S2306943**
**Group:10**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>I used ChatGPT to understand the assignment requirements.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | `['large', 'language', 'models', 'can', 'generate', 'text', 'and', 'answer', 'questions']` | 9 |
| D2 | `['retrieval', 'augmented', 'generation', 'retrieves', 'relevant', 'documents', 'before', 'generating', 'an', 'answer']` | 10 |
| D3 | `['embeddings', 'represent', 'text', 'as', 'numerical', 'vectors']` | 6 |

**avgdl =** 8.3333

**Query terms:** `['how', 'does', 'rag', 'retrieve', 'information']`

**TF, DF and IDF of each query term:**

| Query term | TF in D1 | TF in D2 | TF in D3 | DF | IDF |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

## A1 (b) — BM25 with the stems from README §3

**A1 (b) — BM25 with the stems from README §3**

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | `['large', 'languag', 'model', 'can', 'gener', 'text', 'and', 'answer', 'question']` | 9 |
| D2 | `['retriev', 'augment', 'gener', 'retriev', 'relev', 'document', 'befor', 'gener', 'an', 'answer']` | 10 |
| D3 | `['embed', 'repres', 'text', 'as', 'numer', 'vector']` | 6 |

**avgdl =** 8.3333

**Stemmed query terms:** `['how', 'doe', 'rag', 'retriev', 'inform']`

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 1.0986 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**BM25 of each document**, worked:

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 2 |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 3 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

>In (a), all three documents have a BM25 score of 0 because none of the original query terms appears in the documents. In (b), BM25 retrieves D2 first because stemming changes “retrieve” to “retriev,” which matches words in D2. Therefore, D2 receives a positive BM25 score of 0.6857, while D1 and D3 receive scores of 0.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```


Query = `[0.8, 0.2, 0.5]`

D1 = `[0.7, 0.1, 0.4]`  
D2 = `[0.9, 0.3, 0.6]`  
D3 = `[0.1, 0.8, 0.2]`

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> D2 is the most similar document because it has the highest cosine similarity score of 0.9977.

**Is the ranking the same as BM25's? Compare it with both A1 rankings.**

> The cosine similarity ranking is D2, D1, D3. In A1(a), all BM25 scores were zero, so the ranking was D1, D2, D3 because of the tie order. In A1(b), the ranking was D2, D1, D3. Therefore, the cosine ranking matches the stemmed BM25 ranking in A1(b), but not the original BM25 ranking in A1(a).

**Why can BM25 and embedding retrieval disagree? Point at the words of this query and these documents.**

> BM25 mainly matches exact words, so it can give zero scores when query terms such as “retrieve” and “information” do not appear in the documents. Stemming can help match related word forms, such as “retrieve” and “retrieves” in D2. Embedding retrieval compares vector representations and can identify semantic similarity even when the exact words are different. Therefore, BM25 and embedding retrieval can rank documents differently.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

Error: No connection selected.

## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>I converted all text to lowercase and used regex to split the text into tokens. I did not use stemming or remove stop words.

Error: No connection selected.

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

Error: No connection selected.

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | When was Narxoz University founded? | `["KZ-44"]` | BM25 — the query contains specific keywords such as “Narxoz University” and “founded,” which appear in the relevant chunk. |
| Q-10 | My family has a low income. What financial support can I get as a student? | `["GO-01", "GO-02", "GO-03", "GO-04"]` | Embeddings — the query describes the meaning of the grant without using the exact phrase “need-based grant.” Embeddings can recognize similar meanings even when the wording differs. |

## Results for all ten or more queries: top 3 from each retriever

Error: No connection selected.

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01: What is form GR-305 used for? | GO-08, ..., ... | 1 | GO-08, ..., ... | 1 |
| Q-02: My family is poor. Is there financial help for students like me? | KZ-17, ..., ... | 0 | GO-01, GO-10, GO-20 | 1 |
| Q-03: Өтінішті қашанға дейін тапсыру керек? | GO-14, ..., ... | 1 | GO-14, ..., ... | 1 |
| Q-04: What is the minimum GPA for the grant? | GO-05, GO-02, ... | 0 | GO-02, ..., ... | 1 |
| Q-05: When was Astana renamed Nur-Sultan, and why? | ..., ..., ... | 1 | ..., ..., ... | 1 |
| Q-06: Where did the Soviet Union test its nuclear weapons? | ..., ..., ... | 1 | ..., ..., ... | 1 |
| Q-07: Can the grant be sent to my mother's card? | ..., ..., ... | 0 | GO-15, ..., ... | 1 |
| Q-08: If my grades drop in the autumn, do I still get paid in February? | ..., ..., ... | 1 | ..., ..., ... | 1 |
| Q-09: When was Narxoz University founded? | ..., ..., ... | 1 | ..., ..., ... | 1 |
| Q-10: My family has a low income. What financial support can I get as a student? | ..., ..., ... | 0 | ..., ..., ... | 1 |
| **hit@3 total** | | **6/10** | | **10/10** |

**Both retrievers agree:** which query, and why was it easy?

>Q-01, “What is form GR-305 used for?” Both methods can find GO-08 because the exact form number `GR-305` appears in the relevant chunk.


**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>Neither method clearly wins on the current results. Q-01 and Q-03 are easy for both because they contain specific terms such as `GR-305` and `өтініш`.


**Embeddings did better:** which query, and what could BM25 not match?

>Q-02, “My family is poor. Is there financial help for students like me?” BM25 may not match “financial help” to the exact phrase “need-based study grant,” while embeddings can recognize that the phrases have similar meanings.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

## C1 — the complete prompt, printed before it is sent

\| Part of the input | Tokens (tiktoken \`o200k_base\`) | |---|---:| | System instructions | 43 | | Question | 18 | | Retrieved context | 330 | | Total input | 394 | | \`usage.prompt_tokens\` from the response | 404 |


- Number of retrieved chunks: 3
- Tokens per chunk: GO-04 — 90; GO-03 — 104; GO-15 — 85
- Total context size (tokens / characters): 330 tokens / 1,268 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> They differ because the API also counts message formatting and additional tokens that may not be included in the manual `tiktoken` calculation.

**What exactly does retrieval add to the model's input?**

>Retrieval adds relevant text chunks from the knowledge base to the model's input, giving it specific information to answer the question accurately.

## C2 — three systems, six questions

\| Question | LLM only | BM25 RAG | Embedding RAG | |---|---|---|---| | C-01 | ✗ — does not know | ✓ — 150,000 KZT/year, October and February | ✓ — 150,000 KZT/year, two payments | | C-02 | ✗ — does not know | ✓ — 14 days, form GR-305 | ✓ — 14 days, form GR-305 | | C-03 | ✗ — does not know | ✓ — minimum GPA 2.67 | ✓ — minimum GPA 2.67 | | C-04 | ✗ — does not know | ✓ — October 2017, completion by 2031 | ✓ — October 2017, completion by 2031 | | C-05 | ✗ — does not know | ✓ — founded in 1963 as Almaty Institute of National Economy | ✓ — founded in 1963 as Almaty Institute of National Economy | | C-06 | ✓ — says it does not know the amount | ✓ — amount not specified; ask the department | ✓ — separate master's scheme; ask the department | | \*\*Agrees with \`reference\`\*\* | \*\*1/6\*\* | \*\*6/6\*\* | \*\*6/6\*\* |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>- GO-04 — 20.2192
- GO-02 — 18.7422
- GO-01 — 17.243

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>- GO-04 — 0.5845
- GO-03 — 0.4842
- GO-15 — 0.4506

**3. What was sent to the model** (the context blocks):

>- [1] GO-04 — Grant amounts: income band 2 receives 150,000 KZT per academic year, paid in October and February.
- [2] GO-02 — Eligibility requirements, including minimum GPA 2.67.
- [3] GO-01 — Information about the need-based grant.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Retrieval made the answer more accurate and specific. The model answered that a student in income band 2 receives 150,000 KZT per year in two instalments. EmbeddingGemma also provided the exact amount of each payment: 75,000 KZT.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The LLM-only system could not answer the question because it did not receive the grant rules. This was mainly a retrieval/context problem: the relevant information was missing from its input. Evidence from GO-04: “band 2 150,000/year, two equal payments October/February.

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>- GO-16 — 15.7985
- GO-02 — 14.0587
- GO-06 — 13.8516

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>- GO-02 — 0.6286
- GO-20 — 0.5691
- GO-05 — 0.5451

**3. What was sent to the model** (the context blocks):

>- BM25: GO-16 (eligibility information), GO-02 (current GPA requirement), GO-06 (application form).
- EmbeddingGemma: GO-02 (current GPA requirement), GO-20 (who can apply), GO-05 (archived 2025 rules).

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Both retrieval systems helped the model answer more accurately. BM25 and EmbeddingGemma retrieved GO-02, which states the current minimum GPA requirement of 2.67 out of 4.0.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>EmbeddingGemma also retrieved GO-05, which contains the old 2025 GPA requirement of 3.0. The model needed to distinguish this archived rule from the current rule.

Evidence: GO-02 states that the minimum GPA is “2.67/4.0,” while GO-05 describes the archived 2025 rules. This is mainly a context interpretation problem if the model uses the old value instead of the current one.

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>- GO-16 — 16.9958
- GO-01 — 14.1725
- GO-20 — 13.7202

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>- GO-20 — 0.6664
- GO-16 — 0.6161
- GO-01 — 0.5539

**3. What was sent to the model** (the context blocks):

>- GO-20 — 0.6664
- GO-16 — 0.6161
- GO-01 — 0.5539

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>Both retrieval systems made the answer more grounded. The model explained that the handbook does not specify a grant amount for master's students and recommends contacting the department. EmbeddingGemma ranked GO-20 first, so it provided the most relevant context first.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>The exact grant amount for master's students was not available in the handbook. This was a limitation of the source information, not necessarily a retrieval or generation error.

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 searches for matching words, while embedding retrieval searches for similar meanings using vectors.


**2. When can BM25 perform better than embeddings?**

> BM25 can perform better when the query contains exact keywords, names, codes, or specific terms.

**3. When can embeddings perform better than BM25?**

> Embeddings can perform better when the question uses different words but has the same meaning as the relevant document.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity measures the angle between two vectors to estimate how similar their meanings are, regardless of their lengths.

**5. What information is actually sent to an LLM in a RAG system?**

> The LLM receives the system instructions, the user's question, and the retrieved text chunks.

**6. Does the LLM search the original documents itself?**

> No. The retrieval system searches the documents and sends the selected chunks to the LLM

**7. What happens if retrieval returns an irrelevant chunk?**

> The LLM may give an incorrect or irrelevant answer because it receives misleading context.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> Increasing Top-K can improve RAG by providing more relevant information. However, it can make answers worse by adding irrelevant or outdated chunks. In my experiment, the question was: “How much does a student in income band 2 receive, and when is it paid?” With K = 1, the model retrieved GO-04 and answered 150,000 KZT per year. With K = 3, it also received GO-03 and GO-15, which added information about income bands and payment rules. With K = 8, it received more context, but the answer remained the same. Therefore, K = 3 provided useful additional context without changing the correct answer.

**9. Is a vector database required to build a RAG system?**

> No. A RAG system can use BM25, a simple list of documents, or an ordinary database. A vector database is useful for storing and searching embeddings, but it is not required.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> Embedding retrieval worked better overall. In my ten queries, EmbeddingGemma achieved Hit@3 = 10/10 (1.0), while BM25 achieved Hit@3 = 5/10 (0.5). In C2, both BM25 RAG and Embedding RAG agreed with the reference on 6/6 questions. Therefore, embeddings performed better at retrieving relevant chunks in Part B, while both approaches answered the six C2 questions correctly.

---
# Conclusion

> This homework showed how RAG improves answers by retrieving relevant information and providing it to an LLM. BM25 searches using exact words, while embeddings search by semantic similarity. In my experiments, EmbeddingGemma achieved better Hit@3 than BM25. Both RAG systems answered all six C2 questions correctly, including questions that the LLM alone could not answer. However, retrieval quality matters: irrelevant or outdated chunks can lead to incorrect answers, so the retrieved context should be checked.